# Moebius — Sphere-Congruence Quad Mesh Pipeline

Full pipeline in six stages:
0. **Setup** — imports, env check
1. **Load surface** — B-spline JSON, u/v range, flip options
2. **Initialisation** — sphere radii r(u,v), line congruence l(u,v)
3. **LC optimisation** — minimise l·cu, l·cv; enforce angle with n
4. **Torsal optimisation** — add torsal-plane constraints
5. **Remeshing** — export cross-field, run remesher, load result
6. **Post-optimisation** — sphere fit, support planarity, face regularity
7. **Save results** — OBJ files, sphere panels, state pickle

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

CHAKANA = (ROOT / "Chakana_Geo").resolve()
for sub in ["hanan/src", "kayviz/src"]:
    p = str(CHAKANA / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import matplotlib.pyplot as plt
import kayviz as kv

from moebius.pipeline import (
    MoebiusState, load_surface, sample_selection, compute_init,
    setup_bspline_optimizer, run_bspline_optimizer,
    setup_lc_optimizer, run_lc_optimizer,
    setup_torsal_optimizer, run_torsal_optimizer,
    save_remesh_input, run_remesher, load_remeshed_surface,
    setup_postopt_optimizer, run_postopt_optimizer,
    save_results, load_state,
)
from moebius.utils.bsplines import sample_bspline_surface, bspline_curvatures
from moebius.glyphs import spherical_panels_from_mesh, sphere_centers_radii_from_params
from scipy.interpolate import bisplev

kv.init()

---
## 1 — Load B-spline surface

In [ ]:
# ── parameters (edit these) ───────────────────────────────────────────────────
SURFACE_NAME    = "CMC"          # JSON filename without extension
EXPERIMENT_NAME = "experiment0"

# u/v sampling
U_SAMPLE = 25
V_SAMPLE = 25
INTERVAL_U = [0.0, 1.0]
INTERVAL_V = [0.0, 1.0]

# Orientation options
FLIP_NORMALS = False   # reverse u direction
SWAP_UV      = False   # swap u <-> v
# ─────────────────────────────────────────────────────────────────────────────

state = load_surface(
    SURFACE_NAME, EXPERIMENT_NAME,
    u_sample_num=U_SAMPLE, v_sample_num=V_SAMPLE,
    interval_u=INTERVAL_U, interval_v=INTERVAL_V,
)

# Apply flip/swap if requested
if FLIP_NORMALS:
    cp = state.bspline.controlpoints
    state.bspline.controlpoints = cp[::-1, :, :]
    state.normalsFlipped = True
if SWAP_UV:
    state.bspline = state.bspline.clone()
    state.bspline.controlpoints = state.bspline.controlpoints.swapaxes(0, 1)
    state.uvSwapped = True

print(f"Surface: {SURFACE_NAME}")
print(f"Sampling: {U_SAMPLE}×{V_SAMPLE} on u∈{INTERVAL_U}, v∈{INTERVAL_V}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

def _update_surface(u_min, u_max, v_min, v_max, u_n, v_n):
    if u_min >= u_max or v_min >= v_max:
        return
    sample_selection(state,
                     interval_u=[u_min, u_max],
                     interval_v=[v_min, v_max],
                     u_sample_num=u_n,
                     v_sample_num=v_n)
    V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
    K, H, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
    kv.register_surface_mesh("surface", V, F)
    kv.add_scalar_quantity("surface", "MeanCurvature",     H.ravel(), defined_on="vertices")
    kv.add_scalar_quantity("surface", "GaussianCurvature", K.ravel(), defined_on="vertices")

_sl = dict(continuous_update=False, style={"description_width": "80px"}, layout=widgets.Layout(width="340px"))
u_min_w = widgets.FloatSlider(value=0.0, min=0.0,  max=0.95, step=0.05, description="u min", **_sl)
u_max_w = widgets.FloatSlider(value=1.0, min=0.05, max=1.0,  step=0.05, description="u max", **_sl)
v_min_w = widgets.FloatSlider(value=0.0, min=0.0,  max=0.95, step=0.05, description="v min", **_sl)
v_max_w = widgets.FloatSlider(value=1.0, min=0.05, max=1.0,  step=0.05, description="v max", **_sl)
u_n_w   = widgets.IntSlider(value=U_SAMPLE, min=5, max=60, step=5, description="u samples", **_sl)
v_n_w   = widgets.IntSlider(value=V_SAMPLE, min=5, max=60, step=5, description="v samples", **_sl)

out = widgets.interactive_output(_update_surface, {
    "u_min": u_min_w, "u_max": u_max_w,
    "v_min": v_min_w, "v_max": v_max_w,
    "u_n":   u_n_w,   "v_n":   v_n_w,
})
ui = widgets.VBox([
    widgets.HBox([u_min_w, u_max_w]),
    widgets.HBox([v_min_w, v_max_w]),
    widgets.HBox([u_n_w,   v_n_w]),
])
display(ui, out)
kv.show()

---
## 1b — B-spline optimisation (enforce H ≠ 0)

Slightly deforms the surface control points to ensure the mean curvature H has
a consistent nonzero sign throughout the domain. This makes the sphere-congruence
initialisation r = 1/H well-conditioned.

Skip this cell if the surface already has |H| > 0 everywhere.

In [ ]:
from moebius.pipeline import setup_bspline_optimizer, run_bspline_optimizer

# ── parameters ────────────────────────────────────────────────────────────────
H_THRESH     = 0.05
W_H          = 1.0
W_STEP       = 0.1
BSPLINE_ITER = 30
# ─────────────────────────────────────────────────────────────────────────────

V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
_, H_before, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
print(f"H before  min={H_before.min():.4f}  max={H_before.max():.4f}  "
      f"zero-crossings≈{int((np.diff(np.sign(H_before.ravel())) != 0).sum())}")

setup_bspline_optimizer(state, H_thresh=H_THRESH, w_H=W_H, w_step=W_STEP)
run_bspline_optimizer(state, max_iter=BSPLINE_ITER)

V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
_, H_after, _ = bspline_curvatures(state.bspline, state.u_pts, state.v_pts)
print(f"H after   min={H_after.min():.4f}  max={H_after.max():.4f}  "
      f"zero-crossings≈{int((np.diff(np.sign(H_after.ravel())) != 0).sum())}")

kv.register_surface_mesh("surface", V, F)
kv.add_scalar_quantity("surface", "H_before", H_before.ravel(), defined_on="vertices")
kv.add_scalar_quantity("surface", "H_after",  H_after.ravel(),  defined_on="vertices")
kv.show()

---
## 2 — Sphere-congruence initialisation

In [ ]:
state.init_mode = 0

compute_init(state)

V, F = sample_bspline_surface(state.bspline, state.u_pts, state.v_pts)
r_vals = bisplev(state.u_pts, state.v_pts, state.r_uv)
n_flat = state.n_surf.reshape(-1, 3)
Vc = V + r_vals.ravel()[:, None] * n_flat

kv.register_surface_mesh("sphere_centers_init", Vc, F)
kv.register_surface_mesh("surface", V, F)
kv.add_vector_quantity("surface", "l_init", state.l_init.reshape(-1, 3), defined_on="vertices")
kv.show()

---
## 3a — Line-congruence optimisation

In [ ]:
# ── parameters ────────────────────────────────────────────────────────────────
LC_MAX_ITER  = 60
ANGLE_LC_DEG = 85.0   # max angle l↔n  (degrees; 90 = l must be tangent)
W_LC         = 1.0
W_ORTH       = 1.0
# ─────────────────────────────────────────────────────────────────────────────

state.angle_lc = ANGLE_LC_DEG

setup_lc_optimizer(state, w_lc=W_LC, w_orth=W_ORTH)
run_lc_optimizer(state, max_iter=LC_MAX_ITER)

In [ ]:
l_opt, cp_opt = state.lc_opt.unpack("l", "rij")
state.r_uv[2] = cp_opt
r_vals = bisplev(state.u_pts, state.v_pts, state.r_uv)
Vc = V + r_vals.ravel()[:, None] * n_flat

l_opt   = l_opt.reshape(-1, 3)
angle_n = np.degrees(np.arccos(np.clip(np.einsum("ij,ij->i", l_opt, n_flat), -1, 1)))

kv.register_surface_mesh("surface",        V, F)
kv.register_surface_mesh("sphere_centers", Vc, F)
kv.add_vector_quantity("surface", "l_opt",    l_opt,   defined_on="vertices")
kv.add_scalar_quantity("surface", "angle_l_n", angle_n, defined_on="vertices")
kv.show()

---
## 3b — Torsal optimisation

In [ ]:
# ── parameters ────────────────────────────────────────────────────────────────
TORSAL_MAX_ITER  = 100
ANGLE_TORSAL_DEG = 30.0   # minimum angle between torsal planes
W_TORSAL         = 1.0
W_TANGLE         = 0.5
# ─────────────────────────────────────────────────────────────────────────────

state.angle_torsal = ANGLE_TORSAL_DEG

setup_torsal_optimizer(state, w_lc=W_LC, w_orth=W_ORTH,
                        w_torsal=W_TORSAL, w_tangle=W_TANGLE)
run_torsal_optimizer(state, max_iter=TORSAL_MAX_ITER)

In [ ]:
from moebius.utils.mesh_utils import lc_info_at_grid_points, vec_dot

l_t, u1, v1, u2, v2, nt1, nt2 = state.torsal_opt.unpack(
    "l", "u1", "v1", "u2", "v2", "nt1", "nt2")

l_t  = l_t.reshape(state.u_sample_num, state.v_sample_num, 3)
lc, lu, lv = lc_info_at_grid_points(l_t)
lu = lu.reshape(-1, 3); lv = lv.reshape(-1, 3)
t1 = u1[:, None] * lu + v1[:, None] * lv
t2 = u2[:, None] * lu + v2[:, None] * lv

nt1 = nt1.reshape(-1, 3)
nt2 = nt2.reshape(-1, 3)
torsal_angle = np.degrees(np.arccos(np.clip(np.abs(vec_dot(nt1, nt2)), 0, 1)))

bary = (V[F[:, 0]] + V[F[:, 1]] + V[F[:, 2]] + V[F[:, 3]]) / 4
t1_n = t1 / (np.linalg.norm(t1, axis=1, keepdims=True) + 1e-12)
t2_n = t2 / (np.linalg.norm(t2, axis=1, keepdims=True) + 1e-12)

kv.register_surface_mesh("surface", V, F)
kv.add_vector_quantity("surface", "l", l_t.reshape(-1, 3), defined_on="vertices")
kv.add_scalar_quantity("surface", "torsal_angle", torsal_angle, defined_on="faces")

kv.register_point_cloud("torsal_t1", bary)
kv.add_vector_quantity("torsal_t1", "t1", t1_n)
kv.register_point_cloud("torsal_t2", bary)
kv.add_vector_quantity("torsal_t2", "t2", t2_n)
kv.show()

---
## 4 — Remeshing

In [ ]:
# ── export cross-field for remesher ──────────────────────────────────────────
save_remesh_input(state)
print("Cross-field saved. Now run the remesher externally, or use run_remesher() below.")

In [ ]:
# ── option A: call the remesher from here ────────────────────────────────────
N_FACES_TARGET = 400
# state.remesher_bin = "/path/to/quadRemesher"   # override if needed

ret = run_remesher(state, n_faces=N_FACES_TARGET)
if ret == 0:
    load_remeshed_surface(state)
    kv.register_surface_mesh("remeshed", state.V_remesh, state.F_remesh)
    kv.show()
else:
    print(f"Remesher returned exit code {ret}. ")
    print("You can load the remeshed OBJ manually:")
    print(f"  load_remeshed_surface(state)")

In [ ]:
# ── option B: load previously remeshed OBJ ───────────────────────────────────
# load_remeshed_surface(state)
# kv.register_surface_mesh("remeshed", state.V_remesh, state.F_remesh)
# kv.show()

---
## 5 — Post-optimisation

In [ ]:
# ── parameters ────────────────────────────────────────────────────────────────
POSTOPT_MAX_ITER = 60
W_SPHERE   = 1.0
W_SUPPORT  = 1e-1
W_REG      = 1e-2
# ─────────────────────────────────────────────────────────────────────────────

setup_postopt_optimizer(state,
                         w_sphere=W_SPHERE,
                         w_support=W_SUPPORT,
                         w_reg=W_REG)
run_postopt_optimizer(state, max_iter=POSTOPT_MAX_ITER)

In [ ]:
# ── visualise post-opt result ─────────────────────────────────────────────────
V_opt = state.postopt.unpack("v").reshape(-1, 3)
A, B, C = state.postopt.unpack("A", "B", "C")
B_r = B.reshape(-1, 3)
centers, radii = sphere_centers_radii_from_params(A, B_r, C)

# Spherical panels
pV, pF = spherical_panels_from_mesh(V_opt, state.F_remesh, centers, radii, n_arc=12)

kv.register_surface_mesh("optimised_mesh",  V_opt,  state.F_remesh)
kv.register_surface_mesh("sphere_panels",   pV,     pF)
kv.add_scalar_quantity("optimised_mesh", "sphere_radius", radii, defined_on="faces")
kv.show()

---
## 6 — Save results

In [ ]:
save_results(state)
print("Saved to:", state.save_path)

---
## Utilities — reload & inspect

```python
# Reload a saved state
state = load_state("../notebooks/out/CMC/experiment0/state.pickle")

# Plot energy history
plt.figure(figsize=(8, 4))
plt.semilogy(state.lc_opt.energy,     label="LC")
plt.semilogy(state.torsal_opt.energy, label="Torsal")
plt.legend(); plt.xlabel("Iteration"); plt.ylabel("Energy"); plt.grid()
```